# Generación de datos sintéticos — Sistema de apoyo al triage hospitalario

Este notebook está alineado con la versión corregida de `squema_bd.sql` de 15 tablas.

Genera datos coherentes para:
- Admin, Médico, Administrativo y Paciente.
- Pacientes con y sin cuenta digital.
- Antecedentes y reportes previos.
- Encuentros con triage y médico responsable.
- Observaciones, diagnósticos, notas clínicas y exámenes.
- Medicamentos y prescripciones.
- Facturas y detalle relacionados con el mismo paciente y encuentro.

Las contraseñas se generan con `pwdlib`, para que sean compatibles con el inicio de sesión de FastAPI.


In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv

env_path=Path.cwd()/"pass.env"
loaded=load_dotenv(dotenv_path=env_path,override=True)
print(f"¿Archivo encontrado y cargado?: {loaded}")

PG_CONNECTION_STRING=os.getenv("PG_CONNECTION_STRING")
if not PG_CONNECTION_STRING:
    raise ValueError("Falta PG_CONNECTION_STRING en pass.env")

print("Variables cargadas correctamente.")


¿Archivo encontrado y cargado?: True
Variables cargadas correctamente.


In [2]:
%pip install -q faker "pwdlib[argon2]"


Note: you may need to restart the kernel to use updated packages.


In [3]:
import random
from datetime import date,datetime,timedelta
from zoneinfo import ZoneInfo

import psycopg2
from psycopg2.extras import Json
from faker import Faker
from pwdlib import PasswordHash

fake=Faker("es_CO")
fake.seed_instance(42)
random.seed(42)

TZ=ZoneInfo("America/Bogota")
password_hash=PasswordHash.recommended()

conn=psycopg2.connect(PG_CONNECTION_STRING)
cur=conn.cursor()

cur.execute("SELECT current_database(),current_user;")
bd,usuario=cur.fetchone()
print("Conectado a PostgreSQL")
print("Base de datos:",bd)
print("Usuario:",usuario)


Conectado a PostgreSQL
Base de datos: neondb
Usuario: neondb_owner


## 1. Verificación del esquema

Antes de generar datos, se comprueba que existan las 15 tablas del modelo corregido.


In [4]:
TABLAS_ESPERADAS={
    "roles","usuarios","auditoria_cambios","pacientes","antecedentes",
    "reportes_previos","encuentros","observaciones","diagnosticos",
    "notas_clinicas","examenes","medicamentos","prescripciones",
    "facturas","factura_detalle"
}

cur.execute("""
SELECT table_name
FROM information_schema.tables
WHERE table_schema='public';
""")
tablas_bd={fila[0] for fila in cur.fetchall()}
faltantes=TABLAS_ESPERADAS-tablas_bd

if faltantes:
    raise RuntimeError(f"Faltan tablas del esquema corregido: {sorted(faltantes)}")

print("Las 15 tablas del esquema corregido están disponibles.")


Las 15 tablas del esquema corregido están disponibles.


## 2. Catálogos y utilidades

Se usan valores sintéticos. Los códigos CIE-10 y LOINC incluidos sirven para mantener una estructura clínica coherente.


In [5]:
MUNICIPIOS=["Tumaco","Barbacoas","Ricaurte","Roberto Payán","Magüí Payán","Mosquera","Olaya Herrera","Francisco Pizarro","El Charco","La Tola","Santa Bárbara"]

DIAGNOSTICOS_CIE10=[
    ("J00","Rinofaringitis aguda (resfriado común)"),
    ("I10","Hipertensión esencial"),
    ("E11","Diabetes mellitus tipo 2"),
    ("M54","Dorsalgia"),
    ("R51","Cefalea"),
    ("A09","Diarrea y gastroenteritis de presunto origen infeccioso"),
    ("J45","Asma"),
    ("R10","Dolor abdominal y pélvico"),
    ("S72","Fractura del fémur"),
]

SIGNOS_VITALES_LOINC=[
    ("8310-5","Temperatura corporal",lambda:round(random.uniform(36.0,39.5),1),"Cel"),
    ("8867-4","Frecuencia cardiaca",lambda:random.randint(55,130),"lpm"),
    ("9279-1","Frecuencia respiratoria",lambda:random.randint(12,28),"resp/min"),
    ("8480-6","Presión arterial sistólica",lambda:random.randint(90,160),"mmHg"),
    ("8462-4","Presión arterial diastólica",lambda:random.randint(55,100),"mmHg"),
    ("59408-5","Saturación de oxígeno",lambda:random.randint(88,100),"%"),
]

EXAMENES_CATALOGO=[
    ("718-7","Hemoglobina","laboratorio","g/dL",lambda:round(random.uniform(9.0,16.5),1),28000.0),
    ("6690-2","Leucocitos","laboratorio","10^3/uL",lambda:round(random.uniform(4.0,14.0),1),32000.0),
    ("2345-7","Glucosa","laboratorio","mg/dL",lambda:random.randint(70,220),22000.0),
    ("3094-0","Nitrógeno ureico","laboratorio","mg/dL",lambda:random.randint(7,35),26000.0),
    ("2160-0","Creatinina","laboratorio","mg/dL",lambda:round(random.uniform(0.5,2.2),2),26000.0),
]

MEDICAMENTOS=[
    ("19000001-1","Acetaminofén 500mg","Acetaminofén","500 mg","Tableta","INVIMA 2015M-000101","Vigente",250.0),
    ("19000002-1","Ibuprofeno 400mg","Ibuprofeno","400 mg","Tableta","INVIMA 2016M-000102","Vigente",300.0),
    ("19000003-1","Amoxicilina 500mg","Amoxicilina","500 mg","Cápsula","INVIMA 2014M-000103","Vigente",450.0),
    ("19000004-1","Losartán 50mg","Losartán potásico","50 mg","Tableta","INVIMA 2013M-000104","Vigente",320.0),
    ("19000005-1","Metformina 850mg","Metformina clorhidrato","850 mg","Tableta","INVIMA 2013M-000105","Vigente",280.0),
    ("19000006-1","Omeprazol 20mg","Omeprazol","20 mg","Cápsula","INVIMA 2012M-000106","Vigente",200.0),
    ("19000007-1","Loratadina 10mg","Loratadina","10 mg","Tableta","INVIMA 2011M-000107","Vigente",150.0),
    ("19000008-1","Ácido acetilsalicílico 100mg","Ácido acetilsalicílico","100 mg","Tableta","INVIMA 2010M-000108","Vigente",100.0),
    ("19000009-1","Salbutamol inhalador 100mcg","Salbutamol","100 mcg/dosis","Inhalador","INVIMA 2017M-000109","Vigente",18000.0),
    ("19000010-1","Dexametasona 4mg/ml","Dexametasona","4 mg/ml","Ampolla","INVIMA 2015M-000110","Vigente",900.0),
    ("19000011-1","Solución salina 0.9% 1000ml","Cloruro de sodio","0.9%","Bolsa IV","INVIMA 2009M-000111","Vigente",5200.0),
    ("19000012-1","Insulina glargina 100U/ml","Insulina glargina","100 U/ml","Vial","INVIMA 2018M-000112","Vigente",45000.0),
]

TIPOS_ANTECEDENTE=[
    ("patologico","I10","Hipertensión arterial diagnosticada"),
    ("patologico","E11","Diabetes mellitus tipo 2"),
    ("quirurgico",None,"Apendicectomía"),
    ("alergico",None,"Alergia a penicilina"),
    ("familiar",None,"Antecedente familiar de enfermedad cardiovascular"),
    ("toxico",None,"Tabaquismo activo"),
]

SINTOMAS_PREVIOS=[
    "Dolor torácico intenso","Dificultad para respirar","Fiebre alta persistente",
    "Dolor abdominal severo","Trauma por caída","Convulsiones",
    "Pérdida de conciencia transitoria"
]

documentos_usados=set()

def generar_documento():
    while True:
        doc=random.randint(1_000_000_000,1_199_999_999)
        if doc not in documentos_usados:
            documentos_usados.add(doc)
            return doc

def nombre_apellido():
    return fake.first_name(),fake.last_name()

def telefono():
    return f"+57 3{random.randint(100000000,199999999)}"

def fecha_2026(mes_max=9):
    return datetime(2026,random.randint(1,mes_max),random.randint(1,28),random.randint(0,23),random.randint(0,59),tzinfo=TZ)

print("Catálogos y utilidades listos.")


Catálogos y utilidades listos.


## 3. Limpieza segura de datos sintéticos anteriores

Se conservan los cuatro roles. Las demás tablas se vacían en orden inverso de dependencias para poder volver a ejecutar el notebook.


In [6]:
conn.rollback()

tablas_con_deleted_by=[
    "factura_detalle","facturas","prescripciones","examenes","notas_clinicas",
    "diagnosticos","observaciones","encuentros","reportes_previos","antecedentes",
    "pacientes","usuarios"
]

for tabla in tablas_con_deleted_by:
    cur.execute(f"UPDATE {tabla} SET deleted_by=NULL")

orden_borrado=[
    "auditoria_cambios","factura_detalle","facturas","prescripciones","medicamentos",
    "examenes","notas_clinicas","diagnosticos","observaciones","encuentros",
    "reportes_previos","antecedentes","pacientes","usuarios"
]

for tabla in orden_borrado:
    cur.execute(f"DELETE FROM {tabla}")

secuencias=[
    "auditoria_cambios_id_auditoria_seq","antecedentes_id_antecedente_seq",
    "reportes_previos_id_reporte_seq","encuentros_id_encuentro_seq",
    "observaciones_id_observacion_seq","diagnosticos_id_diagnostico_seq",
    "notas_clinicas_id_nota_seq","examenes_id_examen_seq",
    "prescripciones_id_prescripcion_seq","facturas_id_factura_seq",
    "factura_detalle_id_detalle_seq"
]

for seq in secuencias:
    cur.execute(f"ALTER SEQUENCE {seq} RESTART WITH 1")

conn.commit()
print("Datos anteriores eliminados. Los roles se conservaron.")


Datos anteriores eliminados. Los roles se conservaron.


## 4. Roles y usuarios de personal

Se crean 1 Admin, 4 Médicos y 2 Administrativos.


In [7]:
cur.execute("SELECT id_rol,nombre FROM roles WHERE is_active=TRUE")
ROLES={nombre:id_rol for id_rol,nombre in cur.fetchall()}

esperados={"Admin","Medico","Administrativo","Paciente"}
if set(ROLES)!=esperados:
    raise RuntimeError(f"Roles activos incorrectos: {ROLES}")

def crear_usuario(rol_nombre,nombres,apellidos,prefijo,password):
    doc=generar_documento()
    username=f"{prefijo}.{doc}"
    return {
        "numero_documento_usuario":doc,
        "id_rol":ROLES[rol_nombre],
        "username":username,
        "password_hash":password_hash.hash(password),
        "nombres":nombres,
        "apellidos":apellidos,
        "email":f"{username}@saluddigital.test",
        "telefono":telefono(),
        "password_plano":password
    }

usuarios_staff=[]

n,a=nombre_apellido()
usuarios_staff.append(crear_usuario("Admin",n,a,"admin","Admin123*"))

for i in range(4):
    n,a=nombre_apellido()
    usuarios_staff.append(crear_usuario("Medico",n,a,f"medico{i+1}","Medico123*"))

for i in range(2):
    n,a=nombre_apellido()
    usuarios_staff.append(crear_usuario("Administrativo",n,a,f"administrativo{i+1}","Administrativo123*"))

for u in usuarios_staff:
    cur.execute("""
        INSERT INTO usuarios(
            numero_documento_usuario,id_rol,username,password_hash,
            nombres,apellidos,email,telefono
        )
        VALUES(%s,%s,%s,%s,%s,%s,%s,%s)
    """,(
        u["numero_documento_usuario"],u["id_rol"],u["username"],u["password_hash"],
        u["nombres"],u["apellidos"],u["email"],u["telefono"]
    ))

conn.commit()

usuario_admin=next(u["numero_documento_usuario"] for u in usuarios_staff if u["id_rol"]==ROLES["Admin"])
usuarios_medicos=[u["numero_documento_usuario"] for u in usuarios_staff if u["id_rol"]==ROLES["Medico"]]
usuarios_administrativos=[u["numero_documento_usuario"] for u in usuarios_staff if u["id_rol"]==ROLES["Administrativo"]]

print("Personal creado correctamente.")


Personal creado correctamente.


## 5. Pacientes y cuentas de portal

Se generan 25 pacientes. Aproximadamente el 75% tiene cuenta de usuario con rol `Paciente`.


In [8]:
N_PACIENTES=25
pacientes_data=[]
usuarios_pacientes=[]

for _ in range(N_PACIENTES):
    nombres,apellidos=nombre_apellido()
    doc=generar_documento()
    fecha_nac=date(1945,1,1)+timedelta(days=random.randint(0,75*365))
    edad=(date(2026,9,17)-fecha_nac).days//365
    tipo_documento="TI" if edad<18 else random.choices(["CC","CE"],weights=[0.92,0.08])[0]
    genero_fhir=random.choices(["male","female","other","unknown"],weights=[0.48,0.48,0.02,0.02])[0]
    zona=random.choices(["urbana","rural_dispersa"],weights=[0.6,0.4])[0]
    municipio_residencia=random.choice(MUNICIPIOS)
    tiene_cuenta=random.random()<0.75

    id_usuario=None
    if tiene_cuenta:
        username=f"paciente.{doc}"
        clave="Paciente123*"
        cur.execute("""
            INSERT INTO usuarios(
                numero_documento_usuario,id_rol,username,password_hash,
                nombres,apellidos,email,telefono
            )
            VALUES(%s,%s,%s,%s,%s,%s,%s,%s)
        """,(
            doc,ROLES["Paciente"],username,password_hash.hash(clave),
            nombres,apellidos,f"{username}@saluddigital.test",telefono()
        ))
        id_usuario=doc
        usuarios_pacientes.append({"documento":doc,"username":username,"password":clave})

    paciente={
        "numero_documento_paciente":doc,
        "id_usuario":id_usuario,
        "tipo_documento":tipo_documento,
        "nombres":nombres,
        "apellidos":apellidos,
        "fecha_nacimiento":fecha_nac,
        "genero_fhir":genero_fhir,
        "telefono":telefono(),
        "direccion":fake.street_address(),
        "municipio_residencia":municipio_residencia,
        "zona_residencia":zona
    }
    pacientes_data.append(paciente)

    cur.execute("""
        INSERT INTO pacientes(
            numero_documento_paciente,id_usuario,tipo_documento,nombres,apellidos,
            fecha_nacimiento,genero_fhir,telefono,direccion,municipio_residencia,zona_residencia
        )
        VALUES(%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
    """,(
        paciente["numero_documento_paciente"],paciente["id_usuario"],paciente["tipo_documento"],
        paciente["nombres"],paciente["apellidos"],paciente["fecha_nacimiento"],
        paciente["genero_fhir"],paciente["telefono"],paciente["direccion"],
        paciente["municipio_residencia"],paciente["zona_residencia"]
    ))

conn.commit()
print(f"{len(pacientes_data)} pacientes creados; {len(usuarios_pacientes)} tienen cuenta de portal.")


25 pacientes creados; 19 tienen cuenta de portal.


## 6. Antecedentes


In [9]:
antecedentes_creados=0

for p in pacientes_data:
    if random.random()<0.65:
        for tipo,codigo,descripcion in random.sample(TIPOS_ANTECEDENTE,k=random.randint(1,2)):
            medico=random.choice(usuarios_medicos)
            cur.execute("""
                INSERT INTO antecedentes(id_paciente,tipo,codigo,descripcion,registrado_por)
                VALUES(%s,%s,%s,%s,%s)
            """,(p["numero_documento_paciente"],tipo,codigo,descripcion,medico))
            antecedentes_creados+=1

conn.commit()
print(f"{antecedentes_creados} antecedentes creados.")


23 antecedentes creados.


## 7. Reportes previos


In [10]:
reportes_creados=0

for p in pacientes_data:
    if p["zona_residencia"]=="rural_dispersa" and random.random()<0.75:
        fecha_reporte=fecha_2026()
        signos_alarma=random.random()<0.4
        registrado_por=p["id_usuario"] if p["id_usuario"] else usuario_admin

        cur.execute("""
            INSERT INTO reportes_previos(
                id_paciente,fecha_hora_reporte,sintoma_principal,inicio_sintomas,evolucion,
                signos_alarma_presentes,descripcion_signos_alarma,ubicacion_aproximada,
                municipio_origen,distancia_aproximada_km,tiempo_desplazamiento_min,
                orientacion_inicial,registrado_por
            )
            VALUES(%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
        """,(
            p["numero_documento_paciente"],fecha_reporte,random.choice(SINTOMAS_PREVIOS),
            fecha_reporte-timedelta(hours=random.randint(1,48)),
            "Síntomas en aumento progresivo" if signos_alarma else "Síntomas estables desde el inicio",
            signos_alarma,
            "Dificultad respiratoria, palidez o deterioro general" if signos_alarma else None,
            f"Zona de residencia en {p['municipio_residencia']}",
            p["municipio_residencia"],round(random.uniform(5,140),1),
            random.randint(20,320),"Se orienta acudir al servicio de urgencias",
            registrado_por
        ))
        reportes_creados+=1

conn.commit()
print(f"{reportes_creados} reportes previos creados.")


8 reportes previos creados.


## 8. Encuentros y triage

Cada paciente puede tener varios encuentros históricos, pero como máximo un encuentro activo.


In [11]:
encuentros_creados=[]

for p in pacientes_data:
    n_encuentros=random.randint(1,3)
    fechas=sorted([fecha_2026() for _ in range(n_encuentros)])

    for idx,ingreso in enumerate(fechas):
        es_ultimo=idx==n_encuentros-1
        dejar_activo=es_ultimo and random.random()<0.35

        if dejar_activo:
            estado=random.choice(["en_triage","en_atencion","en_observacion"])
            fin=None
        else:
            estado="finalizado"
            fin=ingreso+timedelta(hours=random.randint(1,12))

        nivel_triage=random.choices([1,2,3,4,5],weights=[0.05,0.15,0.35,0.30,0.15])[0]
        dolor=random.randint(0,10)
        codigo_dx,descripcion_dx=random.choice(DIAGNOSTICOS_CIE10)
        medico_responsable=random.choice(usuarios_medicos)
        creado_por=random.choice(usuarios_administrativos+usuarios_medicos)

        cur.execute("""
            INSERT INTO encuentros(
                id_paciente,fecha_hora_ingreso,fecha_hora_fin,tipo_encuentro,servicio,
                estado,motivo_consulta,observaciones_generales,nivel_triage,
                fecha_hora_triage,dolor_escala,observaciones_triage,clasificado_por,
                clasificacion_automatica,medico_responsable,creado_por
            )
            VALUES(%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
            RETURNING id_encuentro
        """,(
            p["numero_documento_paciente"],ingreso,fin,"urgencias","URGENCIAS",
            estado,descripcion_dx,
            "Paciente valorado al ingreso y asociado al episodio clínico.",
            nivel_triage,ingreso+timedelta(minutes=random.randint(2,20)),
            dolor,"Clasificación clínica inicial realizada por el profesional.",
            medico_responsable,False,medico_responsable,creado_por
        ))

        id_encuentro=cur.fetchone()[0]
        encuentros_creados.append({
            "id_encuentro":id_encuentro,
            "id_paciente":p["numero_documento_paciente"],
            "ingreso":ingreso,
            "estado":estado,
            "medico":medico_responsable,
            "codigo_dx":codigo_dx,
            "descripcion_dx":descripcion_dx
        })

conn.commit()
print(f"{len(encuentros_creados)} encuentros creados.")


53 encuentros creados.


## 9. Observaciones


In [12]:
observaciones_creadas=0

for e in encuentros_creados:
    momento=e["ingreso"]+timedelta(minutes=random.randint(5,30))

    for codigo,nombre,generador,unidad in SIGNOS_VITALES_LOINC:
        cur.execute("""
            INSERT INTO observaciones(
                id_encuentro,tipo_observacion,codigo_loinc,nombre,
                valor_numerico,valor_texto,unidad,fecha_hora_observacion,registrado_por
            )
            VALUES(%s,%s,%s,%s,%s,NULL,%s,%s,%s)
        """,(
            e["id_encuentro"],"signo_vital",codigo,nombre,generador(),
            unidad,momento,e["medico"]
        ))
        observaciones_creadas+=1

conn.commit()
print(f"{observaciones_creadas} observaciones creadas.")


318 observaciones creadas.


## 10. Diagnósticos


In [13]:
diagnosticos_creados=0

for e in encuentros_creados:
    cur.execute("""
        INSERT INTO diagnosticos(
            id_encuentro,codigo_cie10,descripcion,tipo,estado_clinico,
            fecha_diagnostico,registrado_por
        )
        VALUES(%s,%s,%s,%s,%s,%s,%s)
    """,(
        e["id_encuentro"],e["codigo_dx"],e["descripcion_dx"],"principal",
        "active" if e["estado"]!="finalizado" else random.choice(["resolved","inactive"]),
        e["ingreso"]+timedelta(minutes=random.randint(30,90)),e["medico"]
    ))
    diagnosticos_creados+=1

    if random.random()<0.25:
        opciones=[d for d in DIAGNOSTICOS_CIE10 if d[0]!=e["codigo_dx"]]
        codigo2,descripcion2=random.choice(opciones)
        cur.execute("""
            INSERT INTO diagnosticos(
                id_encuentro,codigo_cie10,descripcion,tipo,estado_clinico,
                fecha_diagnostico,registrado_por
            )
            VALUES(%s,%s,%s,'secundario','active',%s,%s)
        """,(
            e["id_encuentro"],codigo2,descripcion2,
            e["ingreso"]+timedelta(minutes=random.randint(40,120)),e["medico"]
        ))
        diagnosticos_creados+=1

conn.commit()
print(f"{diagnosticos_creados} diagnósticos creados.")


70 diagnósticos creados.


## 11. Notas clínicas / evoluciones


In [14]:
notas_creadas=0

for e in encuentros_creados:
    tipos=["valoracion","evolucion"] if e["estado"]=="finalizado" else ["valoracion"]

    for tipo in tipos:
        contenido="Valoración inicial realizada. Se revisan síntomas, antecedentes y signos vitales." if tipo=="valoracion" else "Paciente con evolución clínica documentada durante el encuentro."
        cur.execute("""
            INSERT INTO notas_clinicas(
                id_encuentro,tipo_nota,contenido,fecha_hora,registrado_por
            )
            VALUES(%s,%s,%s,%s,%s)
        """,(
            e["id_encuentro"],tipo,contenido,
            e["ingreso"]+timedelta(minutes=random.randint(30,180)),e["medico"]
        ))
        notas_creadas+=1

conn.commit()
print(f"{notas_creadas} notas clínicas creadas.")


99 notas clínicas creadas.


## 12. Exámenes clínicos


In [15]:
examenes_creados=[]

for e in encuentros_creados:
    if random.random()<0.65:
        seleccion=random.sample(EXAMENES_CATALOGO,k=random.randint(1,2))

        for codigo,nombre,categoria,unidad,generador,precio in seleccion:
            completado=random.random()<0.75
            estado="completed" if completado else "active"
            fecha_solicitud=e["ingreso"]+timedelta(minutes=random.randint(20,90))
            fecha_resultado=fecha_solicitud+timedelta(hours=random.randint(1,6)) if completado else None
            valor=generador() if completado else None
            resultado=f"{valor} {unidad}" if completado else None
            conclusion="Resultado disponible para valoración clínica." if completado else None

            cur.execute("""
                INSERT INTO examenes(
                    id_encuentro,codigo_loinc,nombre,categoria,estado,resultado,conclusion,
                    fecha_solicitud,fecha_resultado,solicitado_por,registrado_resultado_por
                )
                VALUES(%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
                RETURNING id_examen
            """,(
                e["id_encuentro"],codigo,nombre,categoria,estado,resultado,conclusion,
                fecha_solicitud,fecha_resultado,e["medico"],e["medico"] if completado else None
            ))

            examenes_creados.append({
                "id_examen":cur.fetchone()[0],
                "id_encuentro":e["id_encuentro"],
                "nombre":nombre,
                "estado":estado,
                "precio":precio
            })

conn.commit()
print(f"{len(examenes_creados)} exámenes creados.")


55 exámenes creados.


## 13. Catálogo de medicamentos


In [16]:
for cod,nom,principio,conc,forma,registro,estado_cum,precio in MEDICAMENTOS:
    cur.execute("""
        INSERT INTO medicamentos(
            codigo_cum,nombre,principio_activo,concentracion,
            forma_farmaceutica,registro_sanitario,estado_cum,precio_unitario
        )
        VALUES(%s,%s,%s,%s,%s,%s,%s,%s)
        ON CONFLICT(codigo_cum)
        DO UPDATE SET
            nombre=EXCLUDED.nombre,
            principio_activo=EXCLUDED.principio_activo,
            concentracion=EXCLUDED.concentracion,
            forma_farmaceutica=EXCLUDED.forma_farmaceutica,
            registro_sanitario=EXCLUDED.registro_sanitario,
            estado_cum=EXCLUDED.estado_cum,
            precio_unitario=EXCLUDED.precio_unitario,
            is_deleted=FALSE
    """,(cod,nom,principio,conc,forma,registro,estado_cum,precio))

conn.commit()
print(f"{len(MEDICAMENTOS)} medicamentos cargados.")


12 medicamentos cargados.


## 14. Prescripciones


In [17]:
prescripciones_creadas=[]

for e in encuentros_creados:
    if random.random()<0.75:
        seleccion=random.sample(MEDICAMENTOS,k=random.randint(1,2))

        for cod,nom,principio,conc,forma,registro,estado_cum,precio in seleccion:
            cantidad=random.randint(1,3)
            estado=random.choices(["activa","dispensada","anulada"],weights=[0.5,0.4,0.1])[0]

            cur.execute("""
                INSERT INTO prescripciones(
                    id_encuentro,codigo_cum,dosis,frecuencia,via_administracion,
                    cantidad,prescrito_por,fecha_prescripcion,estado
                )
                VALUES(%s,%s,%s,%s,%s,%s,%s,%s,%s)
                RETURNING id_prescripcion
            """,(
                e["id_encuentro"],cod,"1 unidad","Cada 8 horas","Oral",
                cantidad,e["medico"],e["ingreso"]+timedelta(hours=1),estado
            ))

            prescripciones_creadas.append({
                "id_prescripcion":cur.fetchone()[0],
                "id_encuentro":e["id_encuentro"],
                "codigo_cum":cod,
                "nombre":nom,
                "cantidad":cantidad,
                "precio":precio,
                "estado":estado
            })

conn.commit()
print(f"{len(prescripciones_creadas)} prescripciones creadas.")


65 prescripciones creadas.


## 15. Facturas y detalle

Las líneas de factura conservan el mismo `id_encuentro` que la factura y solo referencian prescripciones o exámenes del mismo episodio.


In [18]:
facturas_creadas=[]
detalles_creados=0
VALOR_CONSULTA=45000.0

for e in encuentros_creados:
    if random.random()<0.8:
        administrativo=random.choice(usuarios_administrativos)
        numero_factura=f"FE-2026-{e['id_encuentro']:05d}"
        fecha_emision=e["ingreso"]+timedelta(hours=random.randint(1,12))

        detalles=[{
            "concepto":"Consulta de urgencias",
            "cantidad":1,
            "valor_unitario":VALOR_CONSULTA,
            "id_prescripcion":None,
            "id_examen":None
        }]

        for pr in prescripciones_creadas:
            if pr["id_encuentro"]==e["id_encuentro"] and pr["estado"]!="anulada":
                detalles.append({
                    "concepto":f"Medicamento: {pr['nombre']}",
                    "cantidad":pr["cantidad"],
                    "valor_unitario":pr["precio"],
                    "id_prescripcion":pr["id_prescripcion"],
                    "id_examen":None
                })

        for ex in examenes_creados:
            if ex["id_encuentro"]==e["id_encuentro"]:
                detalles.append({
                    "concepto":f"Examen: {ex['nombre']}",
                    "cantidad":1,
                    "valor_unitario":ex["precio"],
                    "id_prescripcion":None,
                    "id_examen":ex["id_examen"]
                })

        total=sum(d["cantidad"]*d["valor_unitario"] for d in detalles)
        estado=random.choices(["pendiente","pagada","anulada"],weights=[0.3,0.6,0.1])[0]

        cur.execute("""
            INSERT INTO facturas(
                id_paciente,id_encuentro,numero_factura,fecha_emision,
                concepto,total,estado,creado_por
            )
            VALUES(%s,%s,%s,%s,%s,%s,%s,%s)
            RETURNING id_factura
        """,(
            e["id_paciente"],e["id_encuentro"],numero_factura,fecha_emision,
            "Atención de urgencias",total,estado,administrativo
        ))

        id_factura=cur.fetchone()[0]
        facturas_creadas.append({
            "id_factura":id_factura,
            "id_encuentro":e["id_encuentro"],
            "id_paciente":e["id_paciente"],
            "total":total
        })

        for d in detalles:
            valor_total=d["cantidad"]*d["valor_unitario"]

            cur.execute("""
                INSERT INTO factura_detalle(
                    id_factura,id_encuentro,id_prescripcion,id_examen,
                    concepto,cantidad,valor_unitario,valor_total,creado_por
                )
                VALUES(%s,%s,%s,%s,%s,%s,%s,%s,%s)
            """,(
                id_factura,e["id_encuentro"],d["id_prescripcion"],d["id_examen"],
                d["concepto"],d["cantidad"],d["valor_unitario"],valor_total,administrativo
            ))
            detalles_creados+=1

conn.commit()
print(f"{len(facturas_creadas)} facturas y {detalles_creados} detalles creados.")


42 facturas y 134 detalles creados.


## 16. Auditoría sintética mínima

La carga se hace por SQL directo, por eso se crean algunos eventos de prueba para no dejar la auditoría vacía.


In [19]:
cur.execute("DELETE FROM auditoria_cambios")

for e in encuentros_creados[:10]:
    cur.execute("""
        INSERT INTO auditoria_cambios(
            tabla_afectada,registro_id,accion,datos_anteriores,datos_nuevos,realizado_por
        )
        VALUES('encuentros',%s,'CREAR',NULL,%s,%s)
    """,(
        str(e["id_encuentro"]),
        Json({"id_paciente":e["id_paciente"],"estado":e["estado"]}),
        e["medico"]
    ))

for f in facturas_creadas[:5]:
    cur.execute("""
        INSERT INTO auditoria_cambios(
            tabla_afectada,registro_id,accion,datos_anteriores,datos_nuevos,realizado_por
        )
        VALUES('facturas',%s,'CREAR',NULL,%s,%s)
    """,(
        str(f["id_factura"]),
        Json({"id_paciente":f["id_paciente"],"id_encuentro":f["id_encuentro"],"total":f["total"]}),
        random.choice(usuarios_administrativos)
    ))

conn.commit()
print("Auditoría sintética mínima creada.")


Auditoría sintética mínima creada.


## 17. Verificación de integridad


In [20]:
tablas_orden=[
    "roles","usuarios","pacientes","antecedentes","reportes_previos",
    "encuentros","observaciones","diagnosticos","notas_clinicas","examenes",
    "medicamentos","prescripciones","facturas","factura_detalle","auditoria_cambios"
]

print("RESUMEN DE REGISTROS")
for tabla in tablas_orden:
    cur.execute(f"SELECT COUNT(*) FROM {tabla}")
    print(f"{tabla}: {cur.fetchone()[0]}")

validaciones={
    "facturas con paciente distinto al del encuentro":"""
        SELECT COUNT(*)
        FROM facturas f
        JOIN encuentros e ON e.id_encuentro=f.id_encuentro
        WHERE f.id_paciente<>e.id_paciente
    """,
    "detalles asociados a una factura de otro encuentro":"""
        SELECT COUNT(*)
        FROM factura_detalle d
        JOIN facturas f ON f.id_factura=d.id_factura
        WHERE d.id_encuentro<>f.id_encuentro
    """,
    "prescripciones huérfanas":"""
        SELECT COUNT(*)
        FROM prescripciones p
        LEFT JOIN encuentros e ON e.id_encuentro=p.id_encuentro
        WHERE e.id_encuentro IS NULL
    """,
    "observaciones huérfanas":"""
        SELECT COUNT(*)
        FROM observaciones o
        LEFT JOIN encuentros e ON e.id_encuentro=o.id_encuentro
        WHERE e.id_encuentro IS NULL
    """,
    "exámenes huérfanos":"""
        SELECT COUNT(*)
        FROM examenes x
        LEFT JOIN encuentros e ON e.id_encuentro=x.id_encuentro
        WHERE e.id_encuentro IS NULL
    """,
    "pacientes con más de un encuentro activo":"""
        SELECT COUNT(*)
        FROM(
            SELECT id_paciente
            FROM encuentros
            WHERE is_deleted=FALSE AND estado<>'finalizado'
            GROUP BY id_paciente
            HAVING COUNT(*)>1
        )q
    """
}

print("\nVALIDACIONES DE INTEGRIDAD")
errores=0
for nombre,consulta in validaciones.items():
    cur.execute(consulta)
    cantidad=cur.fetchone()[0]
    print(f"{nombre}: {cantidad}")
    errores+=cantidad

if errores:
    raise RuntimeError(f"Se encontraron {errores} inconsistencias.")
else:
    print("Integridad verificada: 0 inconsistencias.")


RESUMEN DE REGISTROS
roles: 4
usuarios: 26
pacientes: 25
antecedentes: 23
reportes_previos: 8
encuentros: 53
observaciones: 318
diagnosticos: 70
notas_clinicas: 99
examenes: 55
medicamentos: 12
prescripciones: 65
facturas: 42
factura_detalle: 134
auditoria_cambios: 15

VALIDACIONES DE INTEGRIDAD
facturas con paciente distinto al del encuentro: 0
detalles asociados a una factura de otro encuentro: 0
prescripciones huérfanas: 0
observaciones huérfanas: 0
exámenes huérfanos: 0
pacientes con más de un encuentro activo: 0
Integridad verificada: 0 inconsistencias.


## 18. Credenciales de prueba


In [21]:
print("CREDENCIALES DE PRUEBA")
print("-"*60)

for u in usuarios_staff:
    rol=next(nombre for nombre,id_rol in ROLES.items() if id_rol==u["id_rol"])
    print(f"{rol:15} | {u['username']:28} | {u['password_plano']}")

if usuarios_pacientes:
    p=usuarios_pacientes[0]
    print(f"{'Paciente':15} | {p['username']:28} | {p['password']}")

print("\nEstas credenciales son únicamente para datos sintéticos de desarrollo.")


CREDENCIALES DE PRUEBA
------------------------------------------------------------
Admin           | admin.1171644825             | Admin123*
Medico          | medico1.1006713773           | Medico123*
Medico          | medico2.1073827621           | Medico123*
Medico          | medico3.1059917677           | Medico123*
Medico          | medico4.1197694986           | Medico123*
Administrativo  | administrativo1.1181650134   | Administrativo123*
Administrativo  | administrativo2.1146395715   | Administrativo123*
Paciente        | paciente.1158509126          | Paciente123*

Estas credenciales son únicamente para datos sintéticos de desarrollo.


In [22]:
cur.close()
conn.close()
print("Conexión cerrada. Datos sintéticos cargados correctamente.")


Conexión cerrada. Datos sintéticos cargados correctamente.
